<span style="color:red; font-weight:bold">The longer and larger the task we give our agent, the more it's performance may start to suffer. </span>

For example, let's say we want an agent to produce a high quality market research report.    
This workflow is quiet long and may require different specialists skills at different times, like proposing an outline, researching on the web, validating your sources, writing the report, and then finally editing.

If you tried to hand this all of to one agent, it would've too many tools to choose from, their context window would begin to overflow with information, and the quality of their output would suffer as a result

A multi-agent system could break down this complex application into multiple specialized agents that work together to solve the problem, rather than relying on a singular agent to handle every step. 

==> The multi-agent system we are gonna be building is a supervisor sub-agent model, which uses a singular orchestrating agent delegating tasks to subagent, with LangChain setting up a system like this is as easy as adding agents as tools for your main agent to call.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os

# Get your key from environment
google_key = os.getenv("GOOGLE_API_KEY")
print(f"Key exists: {bool(google_key)}")

# Create the model
model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash",  # Use flash for free tier
    google_api_key=google_key
)

Key exists: True


In [3]:
def clean_response(response):
    """Extract clean text from agent response"""
    last_msg = response['messages'][-1]
    
    if isinstance(last_msg.content, list):
        # Handle Gemini's list format
        for item in last_msg.content:
            if isinstance(item, dict) and item.get('type') == 'text':
                return item['text']
    else:
        return last_msg.content
    
    return "No text content found"



## Creating subagents

In [4]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Calculate the square of a number"""
    return x ** 2

In [5]:
from langchain.agents import create_agent

# create subagents

subagent_1 = create_agent(
    #model='gpt-5-nano',
    model = model,
    tools=[square_root]
)

subagent_2 = create_agent(
    model= model,
    tools=[square]
)

## Calling subagents

Now we create two tool for our supervising agent.     
Each one allows it to call one of the agents.     

Once we've added those tools to the main agent, it should be able to call each of those agents, which themselves called the tools that we gave them at the start. 

In [6]:
from langchain.messages import HumanMessage

@tool
def call_subagent_1(x: float) -> float:
    """Call subagent 1 in order to calculate the square root of a number"""
    response = subagent_1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent_2(x: float) -> float:
    """Call subagent 2 in order to calculate the square of a number"""
    response = subagent_2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

## Creating the main agent

main_agent = create_agent(
    model= model,
    tools=[call_subagent_1, call_subagent_2],
    system_prompt="You are a helpful assistant who can call subagents to calculate the square root or square of a number.")

## Test

In [7]:
question = "What is the square root of 456?"

response = main_agent.invoke({"messages": [HumanMessage(content=question)]})

In [8]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='What is the square root of 456?', additional_kwargs={}, response_metadata={}, id='d4dcdeb4-498c-48e2-b3dc-120c1b1f4328'),
              AIMessage(content='', additional_kwargs={'function_call': {'name': 'call_subagent_1', 'arguments': '{"x": 456}'}, '__gemini_function_call_thought_signatures__': {'7fa74c67-bdce-426e-aa5b-9ecf8b6eebaf': 'CtoBAb4+9vtc6QaqQLYp88Mxe5U9bTg0BIsS+Ordkxmjdome8yZrRD3INhaSvkw3qfGLE0LlRZpaWCLCGICgwVtzccoOWlvSi1k46lranHugBIpi4KQSbg8hft/HJlEm2wZkVaEmy/I6ZQWlG/LFP6m9EU/wGovKHErAm0UUz4GeWsRQfoe0GbmHH9RJm137T3842fEJvopLbAUyM59crmK97E9A1PZYIV/Ci4GPxoBHxWlIzq/4/XMmxy7O1nQ2fOaSXJD2RsMuXYTrns/nCnjv/asskGSjsLzn1oE='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019c4cf6-7704-7100-a074-cc51c52f55e8-0', tool_calls=[{'name': 'call_subagent_1', 'args': {'x': 456}, 'id': '7fa74c67-bdce-426e-aa5b-9ecf8b6eebaf', 'type': 'tool_call'}], usage_m

In [9]:
print(clean_response(response))

The square root of 456 is 21.354156504062622.


This is definitly an example where tracing is needed to see the full picture, because if we just print out the messages into the notebook we actually can't see what goes on within the subagent. 

We just get the final output from the tool call !

#### LangSmith = Debugger + Inspector + Monitor for AI Agents

It's a separate platform built by the same company (LangChain) specifically for observing, debugging, and evaluating AI agents.

Debugging = Watching the chef cook through the kitchen window                
For AI Agents: Debugging = Seeing WHY your agent did what it did, not just WHAT it did.




In [15]:
# ### Available CSS Properties for <span> tags:

# ## TEXT STYLES:
# <span style="color:red">Red text</span>
# <span style="color:#FF5733">Hex color</span>
# <span style="color:rgb(255, 87, 51)">RGB color</span>
# <span style="font-weight:bold">Bold</span>
# <span style="font-style:italic">Italic</span>
# <span style="text-decoration:underline">Underline</span>
# <span style="text-decoration:line-through">Strikethrough</span>
# <span style="font-weight:bold; font-style:italic">Bold Italic</span>

# ## FONT PROPERTIES:
# <span style="font-family:Arial, sans-serif">Different font</span>
# <span style="font-size:20px">Larger text</span>
# <span style="font-size:small">Small text</span>
# <span style="font-size:large">Large text</span>
# <span style="font-size:150%">Percentage size</span>
# <span style="font-variant:small-caps">Small Caps</span>

# ## ALIGNMENT & POSITION:
# <span style="text-align:center">Center aligned</span>  # Note: needs block element
# <span style="float:right">Floats right</span>
# <span style="vertical-align:super">Superscript</span>
# <span style="vertical-align:sub">Subscript</span>
# <span style="position:relative; top:-5px">Raised text</span>

# ## BACKGROUND & BOX:
# <span style="background-color:yellow">Highlighted</span>
# <span style="background-color:#FFFFCC; padding:5px">With padding</span>
# <span style="border:1px solid black; padding:3px">With border</span>
# <span style="border-radius:5px; background:#f0f0f0; padding:5px">Rounded box</span>

# ## SPACING:
# <span style="letter-spacing:2px">Wide spacing</span>
# <span style="word-spacing:10px">Word spacing</span>
# <span style="line-height:2">Line height</span>

# ## SPECIAL EFFECTS:
# <span style="text-shadow: 2px 2px 4px #000000">Text shadow</span>
# <span style="opacity:0.7">Semi-transparent</span>
# <span style="transform:rotate(-5deg)">Rotated text</span>  # May not work in all Jupyter
# <span style="filter:blur(1px)">Blurred text</span>

# ## CUSTOM CLASSES (Advanced):
# <span class="custom-style">Custom style</span>